# Bari- Hybrid Multimodal Anemia Detection System

## Architecture: Late Fusion (Ensemble) Approach

**Components:**
1. **Tabular Model**: XGBoost for CBC blood test data (14 features)
2. **Visual Model**: EfficientNet-B0 for eye conjunctiva images
3. **Late Fusion**: Soft Voting (weighted probability averaging)

**Target**: 9 diagnosis classes (0-8)


In [ ]:

%pip install --upgrade --force-reinstall numpy xgboost scikit-learn pandas tensorflow Pillow matplotlib seaborn tqdm --quiet

# Import libraries
import numpy as np
import pandas as pd
import pickle
import os
from typing import Tuple, Dict, List
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data processing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# XGBoost
import xgboost as xgb

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('default')
sns.set_palette("husl")

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print(" All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print("\nIf you just ran the pip install above, please RESTART the notebook kernel before executing other cells.")

##  Configuration

In [ ]:

# DATA CONFIGURATION

CSV_PATH = 'data/anemia_tabular/diagnosed_cbc_data_v4 (1).csv'
IMAGE_DIR = '/data/Images'
MODEL_SAVE_DIR = './models'
RESULTS_DIR = './results'

# Create directories
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Feature columns (must match CSV exactly)
FEATURE_COLUMNS = [
    'WBC', 'LYMp', 'NEUTp', 'LYMn', 'NEUTn', 
    'RBC', 'HGB', 'HCT', 'MCV', 'MCH', 
    'MCHC', 'PLT', 'PDW', 'PCT'
]

TARGET_COLUMN = 'Diagnosis'

# Diagnosis classes
DIAGNOSIS_CLASSES = [
    'Healthy',
    'Iron deficiency anemia',
    'Normocytic hypochromic anemia',
    'Normocytic normochromic anemia',
    'Other microcytic anemia',
    'Thrombocytopenia',
    'Leukemia',
    'Leukemia with thrombocytopenia',
    'Macrocytic anemia'
]

# MODEL HYPERPARAMETERS


# XGBoost parameters
XGBOOST_PARAMS = {
    'objective': 'multi:softprob',
    'num_class': 9,
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 200,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'n_jobs': -1,
    'eval_metric': 'mlogloss'
}

# EfficientNet parameters
IMAGE_SIZE = (224, 224, 3)
BATCH_SIZE = 32
EPOCHS = 50
VALIDATION_SPLIT = 0.2

# Fusion weights
FUSION_WEIGHTS = (0.5, 0.5)  # (tabular, visual)

print(" Configuration loaded!")
print(f"CSV Path: {CSV_PATH}")
print(f"Image Directory: {IMAGE_DIR}")
print(f"Number of classes: {len(DIAGNOSIS_CLASSES)}")


#  Part 1: Tabular Model (XGBoost)

Process CBC blood test data using XGBoost.

## 1.1 TabularModel Class Definition

In [ ]:
class TabularModel:
    """XGBoost model for CBC blood test data"""
    
    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        self.model = None
        self.feature_names = FEATURE_COLUMNS
        self.diagnosis_classes = DIAGNOSIS_CLASSES
        
    def load_and_preprocess_data(self, csv_path: str) -> Tuple[np.ndarray, np.ndarray]:
        """Load and preprocess tabular CBC data"""
        print(f"Loading tabular data from: {csv_path}")
        df = pd.read_csv(csv_path)
        
        # Display dataset info
        print(f"\nDataset shape: {df.shape}")
        print(f"\nClass distribution:")
        print(df['Diagnosis'].value_counts())
        
        # Validate feature columns
        missing_features = set(self.feature_names) - set(df.columns)
        if missing_features:
            raise ValueError(f"Missing features in dataset: {missing_features}")
        
        # Extract features and target
        X = df[self.feature_names].values
        y = df['Diagnosis'].values
        
        # Encode labels
        y_encoded = self.label_encoder.fit_transform(y)
        
        print(f"\nLabel mapping:")
        for idx, label in enumerate(self.label_encoder.classes_):
            print(f"  {idx}: {label}")
        
        # Handle missing values
        if np.isnan(X).any():
            print("\nWarning: Found NaN values. Filling with column means...")
            X = pd.DataFrame(X, columns=self.feature_names).fillna(
                pd.DataFrame(X, columns=self.feature_names).mean()
            ).values
        
        # Scale features
        X_scaled = self.scaler.fit_transform(X)
        
        print(f"\nFeature statistics after scaling:")
        print(f"  Mean: {X_scaled.mean(axis=0).mean():.4f}")
        print(f"  Std: {X_scaled.std(axis=0).mean():.4f}")
        
        return X_scaled, y_encoded
    
    def train(self, X_train: np.ndarray, y_train: np.ndarray, 
              X_val: np.ndarray = None, y_val: np.ndarray = None):
        """Train XGBoost classifier"""
        print("\n" + "="*60)
        print("Training XGBoost Tabular Model")
        print("="*60)
        
        self.model = xgb.XGBClassifier(**XGBOOST_PARAMS)
        
        # Prepare evaluation set
        eval_set = [(X_train, y_train)]
        if X_val is not None and y_val is not None:
            eval_set.append((X_val, y_val))
        
        # Train
        self.model.fit(
            X_train, y_train,
            eval_set=eval_set,
            verbose=True
        )
        
        # Feature importance
        importance = self.model.feature_importances_
        feature_importance_df = pd.DataFrame({
            'Feature': self.feature_names,
            'Importance': importance
        }).sort_values('Importance', ascending=False)
        
        print("\nTop 5 Most Important Features:")
        print(feature_importance_df.head())
        
        return feature_importance_df
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Get probability predictions"""
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Get class predictions"""
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)
    
    def save(self, save_dir: str):
        """Save model, scaler, and label encoder"""
        os.makedirs(save_dir, exist_ok=True)
        
        with open(f"{save_dir}/xgboost_model.pkl", 'wb') as f:
            pickle.dump(self.model, f)
        with open(f"{save_dir}/scaler.pkl", 'wb') as f:
            pickle.dump(self.scaler, f)
        with open(f"{save_dir}/label_encoder.pkl", 'wb') as f:
            pickle.dump(self.label_encoder, f)
        
        print(f"\n Tabular model saved to {save_dir}")
    
    def load(self, save_dir: str):
        """Load model, scaler, and label encoder"""
        with open(f"{save_dir}/xgboost_model.pkl", 'rb') as f:
            self.model = pickle.load(f)
        with open(f"{save_dir}/scaler.pkl", 'rb') as f:
            self.scaler = pickle.load(f)
        with open(f"{save_dir}/label_encoder.pkl", 'rb') as f:
            self.label_encoder = pickle.load(f)
        
        print(f" Tabular model loaded from {save_dir}")

print(" TabularModel class defined!")

## 1.2 Load and Preprocess Tabular Data

In [ ]:
# Initialize tabular model
tabular_model = TabularModel()

# Load and preprocess data
X, y = tabular_model.load_and_preprocess_data(CSV_PATH)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=42
)

print(f"\nData split:")
print(f"  Train: {len(X_train)} samples")
print(f"  Val: {len(X_val)} samples")
print(f"  Test: {len(X_test)} samples")

## 1.3 Train Tabular Model

In [ ]:
# Train the model
feature_importance = tabular_model.train(X_train, y_train, X_val, y_val)

## 1.4 Evaluate Tabular Model

In [ ]:
# Evaluate on test set
y_pred_tabular = tabular_model.predict(X_test)
y_proba_tabular = tabular_model.predict_proba(X_test)

acc_tabular = accuracy_score(y_test, y_pred_tabular)
print(f"\n{'='*60}")
print(f"Tabular Model Test Accuracy: {acc_tabular:.4f}")
print(f"{'='*60}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tabular, 
                          target_names=tabular_model.label_encoder.classes_))

## 1.5 Visualize Feature Importance

In [ ]:
# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f" Feature importance plot saved to {RESULTS_DIR}/feature_importance.png")

## 1.6 Save Tabular Model

In [ ]:
# Save the trained model
tabular_model.save(MODEL_SAVE_DIR)


#  Part 2: Visual Model (EfficientNet)

Process eye conjunctiva images using EfficientNet-B0.

## 2.1 VisualModel Class Definition

In [ ]:
class VisualModel:
    """EfficientNet-B0 model for eye conjunctiva images"""
    
    def __init__(self, num_classes: int = 9, input_shape: Tuple[int, int, int] = (224, 224, 3)):
        self.num_classes = num_classes
        self.input_shape = input_shape
        self.model = None
        self.history = None
        
    def build_model(self):
        """Build EfficientNet-B0 with custom classification head"""
        print("\n" + "="*60)
        print("Building EfficientNet-B0 Visual Model")
        print("="*60)
        
        # Load pre-trained EfficientNet-B0
        base_model = EfficientNetB0(
            include_top=False,
            weights='imagenet',
            input_shape=self.input_shape,
            pooling='avg'
        )
        
        # Freeze base model layers initially
        base_model.trainable = False
        
        # Build classification head
        inputs = keras.Input(shape=self.input_shape)
        x = base_model(inputs, training=False)
        x = layers.Dropout(0.3)(x)
        x = layers.Dense(256, activation='relu', 
                        kernel_regularizer=keras.regularizers.l2(0.01))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)
        outputs = layers.Dense(self.num_classes, activation='softmax')(x)
        
        # Create model
        self.model = models.Model(inputs, outputs, name='EfficientNet_Anemia_Detector')
        
        # Compile model
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
        )
        
        print(f"\n Model built successfully!")
        print(f"Total parameters: {self.model.count_params():,}")
        
    def prepare_data_generators(self, data_dir: str, batch_size: int = 32, 
                                validation_split: float = 0.2) -> Tuple:
        """Prepare data generators for training"""
        print(f"\nPreparing data generators from: {data_dir}")
        
        # Data augmentation for training
        train_datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            horizontal_flip=True,
            zoom_range=0.15,
            brightness_range=[0.8, 1.2],
            fill_mode='nearest',
            validation_split=validation_split
        )
        
        # Only rescaling for validation
        val_datagen = ImageDataGenerator(
            rescale=1./255,
            validation_split=validation_split
        )
        
        # Training generator
        train_generator = train_datagen.flow_from_directory(
            data_dir,
            target_size=self.input_shape[:2],
            batch_size=batch_size,
            class_mode='sparse',
            subset='training',
            shuffle=True,
            seed=42
        )
        
        # Validation generator
        val_generator = val_datagen.flow_from_directory(
            data_dir,
            target_size=self.input_shape[:2],
            batch_size=batch_size,
            class_mode='sparse',
            subset='validation',
            shuffle=False,
            seed=42
        )
        
        print(f"\nClass mapping:")
        for class_name, class_idx in train_generator.class_indices.items():
            print(f"  {class_idx}: {class_name}")
        
        print(f"\nTraining samples: {train_generator.samples}")
        print(f"Validation samples: {val_generator.samples}")
        
        return train_generator, val_generator
    
    def train(self, train_generator, val_generator, epochs: int = 50, 
             save_dir: str = './models'):
        """Train the model"""
        print("\n" + "="*60)
        print("Training EfficientNet Visual Model")
        print("="*60)
        
        os.makedirs(save_dir, exist_ok=True)
        
        # Callbacks
        callbacks = [
            ModelCheckpoint(
                filepath=f"{save_dir}/efficientnet_best.h5",
                monitor='val_accuracy',
                mode='max',
                save_best_only=True,
                verbose=1
            ),
            EarlyStopping(
                monitor='val_loss',
                patience=10,
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-7,
                verbose=1
            )
        ]
        
        # Train model
        self.history = self.model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=epochs,
            callbacks=callbacks,
            verbose=1
        )
        
        print("\n Training completed!")
        
    def predict_proba(self, image: np.ndarray) -> np.ndarray:
        """Get probability predictions for a single image"""
        if len(image.shape) == 3:
            image = np.expand_dims(image, axis=0)
        
        # Preprocess
        image = tf.image.resize(image, self.input_shape[:2])
        image = image / 255.0
        
        return self.model.predict(image, verbose=0)
    
    def predict_proba_batch(self, images: np.ndarray) -> np.ndarray:
        """Get probability predictions for a batch of images"""
        images = tf.image.resize(images, self.input_shape[:2])
        images = images / 255.0
        return self.model.predict(images, verbose=0)
    
    def save(self, save_path: str):
        """Save the model"""
        self.model.save(save_path)
        print(f"\n Visual model saved to {save_path}")
    
    def load(self, save_path: str):
        """Load the model"""
        self.model = keras.models.load_model(save_path)
        print(f" Visual model loaded from {save_path}")
    
    def plot_training_history(self, save_path: str = None):
        """Plot training history"""
        if self.history is None:
            print("No training history available")
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Accuracy
        axes[0].plot(self.history.history['accuracy'], label='Train Accuracy')
        axes[0].plot(self.history.history['val_accuracy'], label='Val Accuracy')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Accuracy')
        axes[0].set_title('Model Accuracy')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Loss
        axes[1].plot(self.history.history['loss'], label='Train Loss')
        axes[1].plot(self.history.history['val_loss'], label='Val Loss')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].set_title('Model Loss')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f" Training history plot saved to {save_path}")
        
        plt.show()

print(" VisualModel class defined!")

## 2.2 Build Visual Model

In [ ]:
# Initialize visual model
visual_model = VisualModel(num_classes=9, input_shape=IMAGE_SIZE)
visual_model.build_model()

# Display model summary
visual_model.model.summary()

## 2.3 Prepare Image Data Generators

In [ ]:
# Prepare data generators
train_gen, val_gen = visual_model.prepare_data_generators(
    IMAGE_DIR,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT
)

## 2.4 Visualize Sample Images

In [ ]:
# Visualize some sample images from training set
sample_images, sample_labels = next(train_gen)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i in range(8):
    axes[i].imshow(sample_images[i])
    label_idx = int(sample_labels[i])
    label_name = list(train_gen.class_indices.keys())[list(train_gen.class_indices.values()).index(label_idx)]
    axes[i].set_title(f"{label_name}", fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/sample_images.png', dpi=300, bbox_inches='tight')
plt.show()

print(f" Sample images saved to {RESULTS_DIR}/sample_images.png")

## 2.5 Train Visual Model

⚠️ **Note**: This will take 30-60 minutes on GPU or 2-4 hours on CPU. You can reduce EPOCHS for faster testing.

In [ ]:
# Train the visual model
visual_model.train(
    train_gen, 
    val_gen, 
    epochs=EPOCHS,  # Reduce to 10-20 for faster testing
    save_dir=MODEL_SAVE_DIR
)

## 2.6 Plot Training History

In [ ]:
# Plot training history
visual_model.plot_training_history(f"{RESULTS_DIR}/training_history.png")

## 2.7 Evaluate Visual Model

In [ ]:
# Evaluate on validation set
val_loss, val_accuracy, val_top3 = visual_model.model.evaluate(val_gen)

print(f"\n{'='*60}")
print(f"Visual Model Validation Accuracy: {val_accuracy:.4f}")
print(f"Top-3 Accuracy: {val_top3:.4f}")
print(f"{'='*60}")

## 2.8 Save Visual Model

In [ ]:
# Save the trained model
visual_model.save(f"{MODEL_SAVE_DIR}/efficientnet_model.h5")



#  Part 3: Multimodal Fusion (Late Fusion)

Combine both models using soft voting (probability averaging).

## 3.1 MultimodalAnemiaDetector Class Definition

In [ ]:
class MultimodalAnemiaDetector:
    """Late Fusion Ensemble combining Tabular and Visual models"""
    
    def __init__(self, tabular_model: TabularModel, visual_model: VisualModel,
                 fusion_weights: Tuple[float, float] = (0.5, 0.5)):
        self.tabular_model = tabular_model
        self.visual_model = visual_model
        self.fusion_weights = fusion_weights
        
        # Validate weights
        assert abs(sum(fusion_weights) - 1.0) < 1e-6, "Fusion weights must sum to 1.0"
        
        print("\n" + "="*60)
        print("Multimodal Anemia Detector Initialized")
        print("="*60)
        print(f"Fusion strategy: Soft Voting (probability averaging)")
        print(f"Fusion weights: Tabular={fusion_weights[0]:.2f}, Visual={fusion_weights[1]:.2f}")
    
    def predict_multimodal(self, tabular_input: np.ndarray, 
                          image_input: np.ndarray) -> Dict:
        """Perform multimodal prediction using late fusion"""
        # Ensure correct input shapes
        if len(tabular_input.shape) == 1:
            tabular_input = tabular_input.reshape(1, -1)
        
        # Get predictions from both models
        tabular_probs = self.tabular_model.predict_proba(tabular_input)[0]
        visual_probs = self.visual_model.predict_proba(image_input)[0]
        
        # Late fusion: Weighted average of probabilities
        fused_probs = (self.fusion_weights[0] * tabular_probs + 
                      self.fusion_weights[1] * visual_probs)
        
        # Get prediction
        predicted_class = np.argmax(fused_probs)
        predicted_label = self.tabular_model.label_encoder.inverse_transform([predicted_class])[0]
        confidence = fused_probs[predicted_class]
        
        return {
            'probabilities': fused_probs,
            'predicted_class': predicted_class,
            'predicted_label': predicted_label,
            'confidence': confidence,
            'tabular_probs': tabular_probs,
            'visual_probs': visual_probs
        }
    
    def visualize_prediction(self, tabular_input: np.ndarray, 
                           image_input: np.ndarray,
                           true_label: str = None,
                           save_path: str = None):
        """Visualize a single prediction with probability breakdown"""
        # Get prediction
        result = self.predict_multimodal(tabular_input, image_input)
        
        fig = plt.figure(figsize=(16, 5))
        gs = fig.add_gridspec(1, 3, width_ratios=[1, 1.5, 1.5])
        
        # Plot image
        ax1 = fig.add_subplot(gs[0])
        if len(image_input.shape) == 4:
            image_input = image_input[0]
        ax1.imshow(image_input.astype(np.uint8) if image_input.max() > 1 else image_input)
        ax1.axis('off')
        ax1.set_title('Eye Conjunctiva Image')
        
        # Plot probability comparison
        ax2 = fig.add_subplot(gs[1])
        classes = self.tabular_model.label_encoder.classes_
        x = np.arange(len(classes))
        width = 0.35
        
        ax2.barh(x - width/2, result['tabular_probs'], width, 
                label='Tabular Model', alpha=0.8, color='skyblue')
        ax2.barh(x + width/2, result['visual_probs'], width,
                label='Visual Model', alpha=0.8, color='lightcoral')
        
        ax2.set_yticks(x)
        ax2.set_yticklabels(classes, fontsize=8)
        ax2.set_xlabel('Probability')
        ax2.set_title('Model Predictions Comparison')
        ax2.legend()
        ax2.grid(True, alpha=0.3, axis='x')
        
        # Plot fused probabilities
        ax3 = fig.add_subplot(gs[2])
        colors = ['green' if i == result['predicted_class'] else 'gray' 
                 for i in range(len(classes))]
        ax3.barh(x, result['probabilities'], color=colors, alpha=0.7)
        ax3.set_yticks(x)
        ax3.set_yticklabels(classes, fontsize=8)
        ax3.set_xlabel('Fused Probability')
        ax3.set_title('Final Prediction (Late Fusion)')
        ax3.grid(True, alpha=0.3, axis='x')
        
        # Add prediction text
        pred_text = f"Predicted: {result['predicted_label']}\nConfidence: {result['confidence']:.2%}"
        if true_label:
            pred_text += f"\nTrue Label: {true_label}"
        ax3.text(0.02, 0.98, pred_text, transform=ax3.transAxes,
                verticalalignment='top', bbox=dict(boxstyle='round', 
                facecolor='wheat', alpha=0.5), fontsize=9)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f" Prediction visualization saved to {save_path}")
        
        plt.show()
        
        return result

print(" MultimodalAnemiaDetector class defined!")

## 3.2 Create Multimodal Detector

In [ ]:
# Create multimodal detector
detector = MultimodalAnemiaDetector(
    tabular_model=tabular_model,
    visual_model=visual_model,
    fusion_weights=FUSION_WEIGHTS
)

## 3.3 Test Multimodal Prediction

Make a prediction using both CBC data and eye image.

In [ ]:
# Get a test sample
test_tabular = X_test[0]  # First test sample from tabular data
test_image, test_label = next(val_gen)  # Get a batch from image generator
test_image_single = test_image[0]  # First image from batch

# Make prediction
result = detector.predict_multimodal(test_tabular, test_image_single)

# Display results
print("\n" + "="*60)
print("MULTIMODAL PREDICTION RESULT")
print("="*60)
print(f"\n🔍 Predicted Diagnosis: {result['predicted_label']}")
print(f"📊 Confidence Score: {result['confidence']:.2%}")
print(f"\n📋 Top 3 Most Likely Diagnoses:")

top_3_indices = np.argsort(result['probabilities'])[-3:][::-1]
class_names = tabular_model.label_encoder.classes_

for rank, idx in enumerate(top_3_indices, 1):
    diagnosis = class_names[idx]
    prob = result['probabilities'][idx]
    print(f"  {rank}. {diagnosis:40s} {prob:.2%}")

print("\n" + "="*60)

## 3.4 Visualize Multimodal Prediction

In [ ]:
# Visualize the prediction
detector.visualize_prediction(
    tabular_input=test_tabular,
    image_input=test_image_single,
    true_label=None,  # Set if you know the true label
    save_path=f"{RESULTS_DIR}/multimodal_prediction_example.png"
)


#  Part 4: Inference & Production Use

Use the trained models for predictions on new patients.

## 4.1 Load Trained Models (if starting fresh session)

In [ ]:
# Uncomment to load pre-trained models
'''
# Load tabular model
tabular_model = TabularModel()
tabular_model.load(MODEL_SAVE_DIR)

# Load visual model
visual_model = VisualModel(num_classes=9)
visual_model.load(f"{MODEL_SAVE_DIR}/efficientnet_model.h5")

# Create detector
detector = MultimodalAnemiaDetector(
    tabular_model=tabular_model,
    visual_model=visual_model,
    fusion_weights=(0.5, 0.5)
)
'''
print("Models loaded (uncomment if needed)")

## 4.2 Example: Single Patient Prediction

In [ ]:
# Example patient CBC data (14 features)
example_patient_cbc = np.array([
    6.5,    # WBC
    30.2,   # LYMp
    58.3,   # NEUTp
    2.0,    # LYMn
    3.8,    # NEUTn
    4.2,    # RBC
    11.5,   # HGB - Low (suggests anemia)
    35.2,   # HCT - Low
    85.0,   # MCV
    27.4,   # MCH
    32.7,   # MCHC
    180.0,  # PLT
    12.5,   # PDW
    0.18    # PCT
])

# For demonstration, use a sample image from validation set
example_image, _ = next(val_gen)
example_image = example_image[0]

# Make prediction
result = detector.predict_multimodal(example_patient_cbc, example_image)

print("\n" + "="*60)
print("PATIENT DIAGNOSIS PREDICTION")
print("="*60)
print(f"\n Predicted Diagnosis: {result['predicted_label']}")
print(f" Confidence: {result['confidence']:.2%}")
print(f"\n Probability Distribution:")

for idx, prob in enumerate(result['probabilities']):
    diagnosis = tabular_model.label_encoder.classes_[idx]
    bar = '' * int(prob * 50)
    print(f"  {diagnosis:40s} {bar} {prob:.3f}")

print("\n" + "="*60)

## 4.3 Example: Custom Patient Data

In [ ]:
from PIL import Image

# Function to make prediction with custom data
def predict_patient(cbc_data: dict, image_path: str):
    """
    Make prediction for a patient
    
    Args:
        cbc_data: Dictionary with CBC feature values
        image_path: Path to eye conjunctiva image
    """
    # Prepare tabular input
    tabular_input = np.array([cbc_data[col] for col in FEATURE_COLUMNS])
    
    # Load and prepare image
    img = Image.open(image_path).convert('RGB')
    img = img.resize((224, 224))
    img_array = np.array(img)
    
    # Make prediction
    result = detector.predict_multimodal(tabular_input, img_array)
    
    # Visualize
    detector.visualize_prediction(
        tabular_input=tabular_input,
        image_input=img_array,
        save_path=f"{RESULTS_DIR}/custom_patient_prediction.png"
    )
    
    return result

# Example usage (uncomment and provide your data):
'''
custom_cbc = {
    'WBC': 6.5, 'LYMp': 30.2, 'NEUTp': 58.3,
    'LYMn': 2.0, 'NEUTn': 3.8, 'RBC': 4.2,
    'HGB': 11.5, 'HCT': 35.2, 'MCV': 85.0,
    'MCH': 27.4, 'MCHC': 32.7, 'PLT': 180.0,
    'PDW': 12.5, 'PCT': 0.18
}

result = predict_patient(custom_cbc, 'path/to/patient/eye/image.jpg')
'''

print(" Custom prediction function defined")


#  Part 5: Model Evaluation & Analysis

## 5.1 Confusion Matrix - Tabular Model

In [ ]:
# Compute confusion matrix
cm_tabular = confusion_matrix(y_test, y_pred_tabular)

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(cm_tabular, annot=True, fmt='d', cmap='Blues',
           xticklabels=tabular_model.label_encoder.classes_,
           yticklabels=tabular_model.label_encoder.classes_,
           cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Tabular Model - Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_matrix_tabular.png', dpi=300, bbox_inches='tight')
plt.show()

print(f" Confusion matrix saved to {RESULTS_DIR}/confusion_matrix_tabular.png")

## 5.2 Per-Class Performance Analysis

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Calculate per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred_tabular, average=None
)

# Create DataFrame
metrics_df = pd.DataFrame({
    'Class': tabular_model.label_encoder.classes_,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print("\nPer-Class Performance Metrics:")
print(metrics_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics_df))
width = 0.25

ax.bar(x - width, metrics_df['Precision'], width, label='Precision', alpha=0.8)
ax.bar(x, metrics_df['Recall'], width, label='Recall', alpha=0.8)
ax.bar(x + width, metrics_df['F1-Score'], width, label='F1-Score', alpha=0.8)

ax.set_xlabel('Diagnosis Class')
ax.set_ylabel('Score')
ax.set_title('Per-Class Performance Metrics')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Class'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_class_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n Per-class metrics saved to {RESULTS_DIR}/per_class_metrics.png")

## 5.3 Compare Model Performance

In [ ]:
# Create comparison summary
print("\n" + "="*60)
print("MODEL PERFORMANCE COMPARISON")
print("="*60)

print(f"\n Tabular Model (XGBoost):")
print(f"   Accuracy: {acc_tabular:.4f}")

print(f"\n  Visual Model (EfficientNet):")
print(f"   Validation Accuracy: {val_accuracy:.4f}")
print(f"   Top-3 Accuracy: {val_top3:.4f}")

print(f"\n Multimodal Fusion:")
print(f"   Strategy: Soft Voting (Late Fusion)")
print(f"   Weights: Tabular={FUSION_WEIGHTS[0]}, Visual={FUSION_WEIGHTS[1]}")

print("\n" + "="*60)



#  Part 6: Save & Export

Save all models and results for production use.

In [ ]:
# Create final summary
summary = {
    'model_info': {
        'tabular_model': 'XGBoost',
        'visual_model': 'EfficientNet-B0',
        'fusion_strategy': 'Late Fusion (Soft Voting)',
        'num_classes': 9
    },
    'performance': {
        'tabular_accuracy': float(acc_tabular),
        'visual_accuracy': float(val_accuracy),
        'visual_top3_accuracy': float(val_top3)
    },
    'configuration': {
        'fusion_weights': FUSION_WEIGHTS,
        'image_size': IMAGE_SIZE,
        'batch_size': BATCH_SIZE,
        'epochs_trained': EPOCHS
    },
    'diagnosis_classes': DIAGNOSIS_CLASSES,
    'feature_columns': FEATURE_COLUMNS
}

# Save summary
import json

with open(f'{MODEL_SAVE_DIR}/model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(" Model summary saved!")
print(f"\nSaved files in {MODEL_SAVE_DIR}/:")
for file in os.listdir(MODEL_SAVE_DIR):
    print(f"  - {file}")

print(f"\nResults saved in {RESULTS_DIR}/:")
for file in os.listdir(RESULTS_DIR):
    print(f"  - {file}")